# TAHAP 02 — Dataset Design & Data Preparation

## Project
EduPath Career Assistant: Chatbot Rekomendasi Program Studi S1 dan Roadmap Belajar untuk Siswa SMA/SMK/MA Berbasis NLP, Intent Classification, dan Recommender System.

## Tujuan
Notebook ini digunakan untuk:
1. Mendesain struktur dataset utama.
2. Membuat template dataset dalam format CSV.
3. Mengisi contoh data awal.
4. Melakukan validasi awal dataset.
5. Mengekspor dataset ke folder data/raw dan data/interim.
6. Menyiapkan dataset agar siap digunakan pada tahap text preprocessing dan modeling.

In [15]:
from pathlib import Path
import pandas as pd
import re
from datetime import datetime
from IPython.display import display

# Deteksi lokasi notebook
CURRENT_DIR = Path.cwd()

# Jika notebook dijalankan dari folder notebooks, naik satu level ke root project
if CURRENT_DIR.name.lower() == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

DATA_RAW_DIR = PROJECT_ROOT / "data" / "raw"
DATA_INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
REPORT_DIR = PROJECT_ROOT / "reports" / "data_quality"

# Membuat folder jika belum tersedia
for path in [DATA_RAW_DIR, DATA_INTERIM_DIR, REPORT_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print("CURRENT_DIR       :", CURRENT_DIR)
print("PROJECT_ROOT      :", PROJECT_ROOT)
print("DATA_RAW_DIR      :", DATA_RAW_DIR)
print("DATA_INTERIM_DIR  :", DATA_INTERIM_DIR)
print("REPORT_DIR        :", REPORT_DIR)

CURRENT_DIR       : D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA-MINING\TUGAS\UAS\edupath-career-assitant\notebooks
PROJECT_ROOT      : D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA-MINING\TUGAS\UAS\edupath-career-assitant
DATA_RAW_DIR      : D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA-MINING\TUGAS\UAS\edupath-career-assitant\data\raw
DATA_INTERIM_DIR  : D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA-MINING\TUGAS\UAS\edupath-career-assitant\data\interim
REPORT_DIR        : D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA-MINING\TUGAS\UAS\edupath-career-assitant\reports\data_quality


## Definisi Kolom Dataset

Pada bagian ini dibuat daftar kolom untuk `6` dataset utama:
1. Dataset intent classification.
2. Dataset program studi S1.
3. Dataset karier.
4. Dataset skill awal.
5. Dataset roadmap belajar.
6. Dataset normalisasi bahasa.

In [16]:
intent_columns = [
    "utterance_id", "utterance", "intent_label", "bahasa", "register",
    "jenjang_user", "kelas", "minat_keywords", "mapel_favorit",
    "hobi_keywords", "gaya_belajar", "tujuan_karier_keywords",
    "target_program_id", "source_type", "confidence_label", "catatan"
]

program_columns = [
    "program_id", "nama_program_studi", "jenjang", "rumpun_ilmu",
    "bidang_keilmuan", "deskripsi_singkat", "minat_cocok",
    "mapel_relevan", "hobi_relevan", "gaya_belajar_cocok",
    "karakteristik_cocok", "tingkat_matematika", "tingkat_coding",
    "tingkat_komunikasi", "tingkat_desain", "prospek_karier_id_list",
    "skill_id_list", "keyword_rekomendasi", "catatan"
]

karier_columns = [
    "karier_id", "nama_karier", "bidang_karier", "deskripsi_singkat",
    "tugas_utama", "skill_teknis", "skill_non_teknis", "tools_umum",
    "program_studi_relevan_id_list", "prospek_industri", "keyword_karier",
    "catatan"
]

skill_columns = [
    "skill_id", "nama_skill", "kategori_skill", "deskripsi_singkat",
    "level_awal", "program_studi_relevan_id_list", "karier_relevan_id_list",
    "prasyarat", "sumber_belajar_awal", "estimasi_waktu_belajar",
    "keyword_skill", "catatan"
]

roadmap_columns = [
    "roadmap_id", "program_id", "fase", "urutan_fase", "durasi_rekomendasi",
    "tujuan_fase", "materi_pokok", "aktivitas_praktik",
    "output_portofolio", "tools_umum", "indikator_capaian", "catatan"
]

normalisasi_columns = [
    "norm_id", "kata_input", "kata_baku", "bahasa_sumber",
    "kategori", "contoh_kalimat", "prioritas", "catatan"
]

print("Jumlah kolom intent      :", len(intent_columns))
print("Jumlah kolom prodi       :", len(program_columns))
print("Jumlah kolom karier      :", len(karier_columns))
print("Jumlah kolom skill       :", len(skill_columns))
print("Jumlah kolom roadmap     :", len(roadmap_columns))
print("Jumlah kolom normalisasi :", len(normalisasi_columns))

Jumlah kolom intent      : 16
Jumlah kolom prodi       : 19
Jumlah kolom karier      : 12
Jumlah kolom skill       : 12
Jumlah kolom roadmap     : 12
Jumlah kolom normalisasi : 8


## Contoh Data Awal

Data berikut adalah seed dataset awal. Data ini belum dimaksudkan sebagai dataset final, tetapi sebagai fondasi awal untuk:
1. Menguji struktur dataset.
2. Menguji relasi antarfile.
3. Menguji validasi dataset.
4. Menjadi dasar ekspansi dataset pada tahap berikutnya.

Pada tahap final, dataset intent dapat diperbanyak menjadi ribuan baris menggunakan variasi kalimat formal, non-formal, Jawa umum, dan Sunda umum.

In [17]:
intent_data = [
    ["UTT001", "Saya suka matematika dan ingin bekerja di bidang data, program studi apa yang cocok?", "rekomendasi_prodi", "id_formal", "formal", "SMA", "12", "matematika|data|analisis", "Matematika|Informatika", "membuat grafik|menganalisis data", "logis|analitis", "data analyst", "PRG003", "seed_manual", 0.95, "Contoh formal"],
    ["UTT002", "Aku seneng ngoding sama statistik, cocoknya kuliah apa ya?", "rekomendasi_prodi", "id_nonformal", "nonformal", "SMK", "12", "coding|statistik", "Matematika|Produktif RPL", "ngoding|eksperimen aplikasi", "praktik|logis", "programmer|data scientist", "PRG001", "seed_manual", 0.90, "Contoh non-formal"],
    ["UTT003", "Aku pengin dadi analis data, jurusan sing pas apa?", "rekomendasi_prodi", "jawa_umum", "nonformal", "SMA", "12", "data|analisis", "Matematika", "olah data", "analitis", "data analyst", "PRG003", "seed_manual", 0.85, "Contoh Jawa umum"],
    ["UTT004", "Abdi resep komputer jeung angka, prodi naon nu cocog?", "rekomendasi_prodi", "sunda_umum", "nonformal", "SMA", "12", "komputer|angka", "Matematika|Informatika", "komputer", "logis", "software engineer|data analyst", "PRG001", "seed_manual", 0.85, "Contoh Sunda umum"],
    ["UTT005", "Kalau ingin masuk Teknik Informatika, saya harus belajar apa dulu?", "roadmap_belajar", "id_formal", "formal", "SMA", "12", "informatika|pemrograman", "Informatika", "ngoding", "praktik", "software engineer", "PRG001", "seed_manual", 0.95, "Pertanyaan roadmap"],
    ["UTT006", "Lulusan Sistem Informasi biasanya bisa bekerja sebagai apa?", "prospek_karier", "id_formal", "formal", "SMA", "11", "bisnis|teknologi", "Ekonomi|Informatika", "organisasi|membuat rencana", "komunikatif|analitis", "business analyst", "PRG002", "seed_manual", 0.95, "Pertanyaan karier"],
    ["UTT007", "Skill awal untuk jadi data analyst itu apa saja?", "skill_awal", "id_nonformal", "nonformal", "SMK", "12", "data|analisis", "Matematika", "excel|dashboard", "praktik", "data analyst", "PRG003", "seed_manual", 0.95, "Pertanyaan skill"],
    ["UTT008", "Apa perbedaan Teknik Informatika dan Sistem Informasi?", "info_program_studi", "id_formal", "formal", "SMA", "12", "komputer|bisnis", "Informatika|Ekonomi", "membandingkan jurusan", "membaca|analitis", "", "", "seed_manual", 0.90, "Pertanyaan informasi"],
    ["UTT009", "Halo, saya mau konsultasi jurusan kuliah.", "sapaan", "id_formal", "formal", "SMA", "12", "", "", "", "", "", "", "seed_manual", 0.99, "Sapaan awal"],
    ["UTT010", "Saya bingung banget dan belum tahu minat saya apa.", "klarifikasi_minat", "id_nonformal", "nonformal", "SMA", "11", "", "", "", "reflektif", "", "", "seed_manual", 0.90, "Perlu tanya balik"],
    ["UTT011", "Bisa bantu buat roadmap belajar selama 6 bulan untuk Sains Data?", "roadmap_belajar", "id_formal", "formal", "SMA", "12", "data|statistik|machine learning", "Matematika", "analisis data", "terstruktur", "data scientist", "PRG003", "seed_manual", 0.95, "Roadmap Sains Data"],
    ["UTT012", "asdf jurusan awan biru kerja apa", "fallback", "id_nonformal", "nonformal", "SMA", "12", "", "", "", "", "", "", "seed_manual", 0.70, "Contoh fallback sederhana"],
]

program_data = [
    ["PRG001", "Teknik Informatika", "S1", "Komputer", "Software Engineering dan Komputasi", "Mempelajari pemrograman, algoritma, basis data, jaringan, dan pengembangan perangkat lunak.", "coding|logika|komputer|aplikasi", "Matematika|Informatika", "ngoding|membuat aplikasi|problem solving", "praktik|logis|eksperimen", "teliti|suka memecahkan masalah", "tinggi", "tinggi", "sedang", "rendah", "KAR003", "SKL001|SKL002|SKL004", "informatika|pemrograman|software|aplikasi|algoritma", "Seed data"],
    ["PRG002", "Sistem Informasi", "S1", "Komputer", "Sistem Bisnis dan Teknologi Informasi", "Mempelajari integrasi teknologi, proses bisnis, analisis kebutuhan, dan manajemen sistem informasi.", "bisnis|teknologi|analisis|manajemen", "Ekonomi|Informatika|Matematika", "organisasi|membuat diagram|analisis proses", "komunikatif|analitis|praktik", "mampu menjembatani bisnis dan IT", "sedang", "sedang", "tinggi", "rendah", "KAR005|KAR001", "SKL002|SKL003|SKL005", "sistem informasi|bisnis|analisis sistem|erp|business analyst", "Seed data"],
    ["PRG003", "Sains Data", "S1", "Komputer", "Data Science dan Analitik", "Mempelajari statistika, pemrograman, machine learning, visualisasi data, dan pengambilan keputusan berbasis data.", "data|statistik|machine learning|analisis", "Matematika|Statistika|Informatika", "membuat grafik|menganalisis angka|eksperimen model", "analitis|terstruktur|eksperimen", "teliti|suka pola dan data", "tinggi", "tinggi", "sedang", "rendah", "KAR001|KAR002", "SKL001|SKL002|SKL003|SKL006", "sains data|data science|statistik|machine learning|dashboard", "Seed data"],
    ["PRG004", "Statistika", "S1", "Matematika dan Sains", "Statistika Terapan", "Mempelajari pengumpulan data, probabilitas, pemodelan statistik, survei, dan analisis kuantitatif.", "angka|riset|survei|analisis", "Matematika", "olah data|membuat survei|riset", "analitis|terstruktur", "teliti dan kuat secara kuantitatif", "tinggi", "sedang", "sedang", "rendah", "KAR001|KAR002", "SKL003|SKL006", "statistika|probabilitas|survei|analisis kuantitatif", "Seed data"],
    ["PRG005", "Desain Komunikasi Visual", "S1", "Seni dan Desain", "Desain Visual dan Komunikasi Digital", "Mempelajari desain grafis, branding, ilustrasi, UI, komunikasi visual, dan media digital.", "desain|visual|kreatif|media", "Seni Budaya|TIK", "menggambar|editing|membuat konten", "visual|kreatif|praktik", "kreatif dan peka estetika", "rendah", "rendah", "tinggi", "tinggi", "KAR004", "SKL007", "dkv|desain|visual|ui ux|branding|konten", "Seed data"],
]

karier_data = [
    ["KAR001", "Data Analyst", "Data dan Analitik", "Menganalisis data untuk menghasilkan insight bisnis.", "membersihkan data|membuat dashboard|menyusun laporan insight", "Excel|SQL|Python dasar|visualisasi data", "komunikasi|critical thinking|storytelling", "Excel|SQL|Power BI|Looker Studio|Python", "PRG002|PRG003|PRG004", "tinggi", "data analyst|dashboard|sql|insight", "Seed data"],
    ["KAR002", "Data Scientist", "Data dan AI", "Membangun model analitik dan machine learning untuk prediksi atau rekomendasi.", "EDA|feature engineering|modeling|evaluasi model", "Python|SQL|Machine Learning|Statistika", "problem solving|eksperimen|komunikasi", "Python|Jupyter|Scikit-learn|SQL", "PRG003|PRG004", "tinggi", "data scientist|machine learning|model prediksi", "Seed data"],
    ["KAR003", "Software Engineer", "Software Development", "Mengembangkan aplikasi web, mobile, API, dan sistem perangkat lunak.", "coding|testing|debugging|deploy aplikasi", "Algoritma|Git|Database|Backend|Frontend", "kolaborasi|ketelitian|problem solving", "VS Code|Git|GitHub|PostgreSQL", "PRG001", "tinggi", "software engineer|programmer|developer", "Seed data"],
    ["KAR004", "UI/UX Designer", "Desain Produk Digital", "Merancang tampilan dan pengalaman pengguna aplikasi.", "user research|wireframe|prototype|testing", "Figma|design thinking|user flow", "empati|komunikasi|kreativitas", "Figma|FigJam|Miro", "PRG005", "sedang", "ui ux|designer|prototype|figma", "Seed data"],
    ["KAR005", "Business Analyst", "Bisnis dan Teknologi", "Menganalisis kebutuhan bisnis dan menerjemahkannya menjadi solusi sistem.", "requirement gathering|process mapping|dokumentasi", "Analisis proses|SQL dasar|UML|komunikasi bisnis", "komunikasi|negosiasi|analitis", "Draw.io|Jira|Excel|SQL", "PRG002", "tinggi", "business analyst|requirement|proses bisnis", "Seed data"],
]

skill_data = [
    ["SKL001", "Dasar Pemrograman Python", "technical", "Memahami variabel, tipe data, percabangan, perulangan, fungsi, dan struktur data sederhana.", "pemula", "PRG001|PRG003", "KAR002|KAR003", "Logika dasar", "Dokumentasi Python|tutorial dasar Python|latihan coding", "4-6 minggu", "python|coding|programming", "Seed data"],
    ["SKL002", "Dasar Basis Data dan SQL", "technical", "Memahami tabel, relasi, query SELECT, filter, join, agregasi, dan pengelolaan data sederhana.", "pemula", "PRG001|PRG002|PRG003", "KAR001|KAR002|KAR003|KAR005", "Logika data", "Latihan SQL sederhana|SQLite|PostgreSQL dasar", "3-5 minggu", "sql|database|query", "Seed data"],
    ["SKL003", "Analisis Data Dasar", "technical", "Memahami cara membaca data, statistik deskriptif, pola data, dan insight sederhana.", "pemula", "PRG002|PRG003|PRG004", "KAR001|KAR005", "Matematika dasar", "Excel|Google Sheets|dataset sederhana", "3-4 minggu", "analisis data|statistik deskriptif|insight", "Seed data"],
    ["SKL004", "Algoritma dan Struktur Data Dasar", "technical", "Memahami problem solving, array, list, searching, dan sorting dasar.", "pemula", "PRG001", "KAR003", "Pemrograman dasar", "Latihan algoritma dasar|pseudocode", "6-8 minggu", "algoritma|struktur data|problem solving", "Seed data"],
    ["SKL005", "Komunikasi dan Analisis Kebutuhan", "soft_skill", "Mampu menggali kebutuhan, membuat pertanyaan, menyusun catatan, dan menjelaskan solusi.", "pemula", "PRG002", "KAR005", "Kemampuan komunikasi dasar", "Latihan wawancara kebutuhan|membuat user story", "2-4 minggu", "komunikasi|requirement|user story", "Seed data"],
    ["SKL006", "Statistika Dasar", "technical", "Memahami mean, median, modus, variansi, probabilitas dasar, dan interpretasi grafik.", "pemula", "PRG003|PRG004", "KAR001|KAR002", "Matematika dasar", "Latihan statistik dengan Excel/Python", "4-6 minggu", "statistika|probabilitas|mean|median", "Seed data"],
    ["SKL007", "Dasar Desain Visual", "technical", "Memahami warna, tipografi, layout, komposisi, dan prinsip komunikasi visual.", "pemula", "PRG005", "KAR004", "Minat visual", "Canva|Figma|latihan poster sederhana", "3-5 minggu", "desain visual|layout|tipografi", "Seed data"],
]

roadmap_data = [
    ["RMP001", "PRG001", "Fondasi", 1, "1-2 bulan", "Memahami logika dan dasar pemrograman.", "logika algoritma|Python dasar|Git dasar", "membuat program kalkulator sederhana|latihan percabangan dan perulangan", "repo GitHub berisi latihan Python dasar", "Python|VS Code|GitHub", "Mampu membuat program sederhana dan menjelaskan alurnya.", "Seed data"],
    ["RMP002", "PRG001", "Penguatan", 2, "2-3 bulan", "Memahami basis data dan pengembangan aplikasi sederhana.", "SQL dasar|CRUD|HTML CSS dasar|API sederhana", "membuat aplikasi catatan sederhana dengan database", "aplikasi CRUD sederhana", "SQLite|PostgreSQL|Flask/FastAPI", "Mampu membuat aplikasi kecil berbasis database.", "Seed data"],
    ["RMP003", "PRG002", "Fondasi", 1, "1-2 bulan", "Memahami konsep bisnis dan sistem informasi.", "proses bisnis|flowchart|UML dasar", "membuat diagram proses pendaftaran siswa", "dokumen analisis proses bisnis", "Draw.io|Google Docs", "Mampu memetakan proses bisnis sederhana.", "Seed data"],
    ["RMP004", "PRG002", "Penguatan", 2, "2-3 bulan", "Memahami database, analisis kebutuhan, dan dashboard dasar.", "SQL dasar|requirement gathering|dashboard", "membuat user story dan dashboard sederhana", "dokumen kebutuhan dan dashboard", "SQL|Excel|Looker Studio", "Mampu menghubungkan kebutuhan bisnis dan data.", "Seed data"],
    ["RMP005", "PRG003", "Fondasi", 1, "1-2 bulan", "Memahami dasar Python, statistik, dan pengolahan data.", "Python dasar|Pandas|statistika deskriptif", "membersihkan dataset sederhana dan membuat grafik", "notebook EDA sederhana", "Python|Jupyter|Pandas", "Mampu melakukan EDA sederhana.", "Seed data"],
    ["RMP006", "PRG003", "Penguatan", 2, "2-3 bulan", "Memahami machine learning dasar dan evaluasi model.", "supervised learning|train test split|accuracy|F1-score", "membuat model klasifikasi sederhana", "notebook model klasifikasi", "Scikit-learn|Jupyter", "Mampu melatih dan mengevaluasi model dasar.", "Seed data"],
]

normalisasi_data = [
    ["NRM001", "gw", "saya", "id_nonformal", "pronomina", "gw suka matematika", 1, "Bahasa gaul umum"],
    ["NRM002", "gue", "saya", "id_nonformal", "pronomina", "gue bingung pilih jurusan", 1, "Bahasa gaul umum"],
    ["NRM003", "aku", "saya", "id_nonformal", "pronomina", "aku mau kuliah apa ya", 2, "Dipakai untuk variasi non-formal"],
    ["NRM004", "pgn", "ingin", "id_nonformal", "singkatan", "pgn jadi programmer", 1, "Singkatan"],
    ["NRM005", "pingin", "ingin", "id_nonformal", "variasi", "pingin kerja di data", 1, "Variasi kata"],
    ["NRM006", "ngoding", "pemrograman", "id_nonformal", "istilah", "suka ngoding", 1, "Istilah populer"],
    ["NRM007", "jurusan", "program studi", "id_nonformal", "istilah", "jurusan apa yang cocok", 2, "Istilah umum siswa"],
    ["NRM008", "matkul", "mata kuliah", "id_nonformal", "singkatan", "matkul informatika apa saja", 2, "Singkatan kampus"],
    ["NRM009", "seneng", "suka", "jawa_umum", "kata_umum", "aku seneng komputer", 1, "Jawa umum"],
    ["NRM010", "pengin", "ingin", "jawa_umum", "kata_umum", "aku pengin dadi analis data", 1, "Jawa umum"],
    ["NRM011", "dadi", "menjadi", "jawa_umum", "kata_umum", "pengin dadi programmer", 1, "Jawa umum"],
    ["NRM012", "sing", "yang", "jawa_umum", "kata_umum", "jurusan sing cocok", 1, "Jawa umum"],
    ["NRM013", "opo", "apa", "jawa_umum", "kata_tanya", "opo jurusan sing pas", 1, "Jawa umum"],
    ["NRM014", "resep", "suka", "sunda_umum", "kata_umum", "abdi resep komputer", 1, "Sunda umum"],
    ["NRM015", "hoyong", "ingin", "sunda_umum", "kata_umum", "abdi hoyong janten analis", 1, "Sunda umum"],
    ["NRM016", "janten", "menjadi", "sunda_umum", "kata_umum", "hoyong janten programmer", 1, "Sunda umum"],
    ["NRM017", "naon", "apa", "sunda_umum", "kata_tanya", "prodi naon nu cocog", 1, "Sunda umum"],
    ["NRM018", "kumaha", "bagaimana", "sunda_umum", "kata_tanya", "kumaha cara belajar data", 1, "Sunda umum"],
]

In [18]:
dfs = {
    "intent_dataset_raw": pd.DataFrame(intent_data, columns=intent_columns),
    "program_studi_s1_raw": pd.DataFrame(program_data, columns=program_columns),
    "karier_raw": pd.DataFrame(karier_data, columns=karier_columns),
    "skill_awal_raw": pd.DataFrame(skill_data, columns=skill_columns),
    "roadmap_belajar_raw": pd.DataFrame(roadmap_data, columns=roadmap_columns),
    "normalisasi_bahasa_raw": pd.DataFrame(normalisasi_data, columns=normalisasi_columns),
}

dataset_overview = []

for name, df in dfs.items():
    dataset_overview.append({
        "dataset": name,
        "rows": df.shape[0],
        "columns": df.shape[1]
    })

dataset_overview_df = pd.DataFrame(dataset_overview)
display(dataset_overview_df)

,dataset,rows,columns
0,intent_dataset_raw,12,16
1,program_studi_s1_raw,5,19
2,karier_raw,5,12
3,skill_awal_raw,7,12
4,roadmap_belajar_raw,6,12
5,normalisasi_bahasa_raw,18,8


In [19]:
print("Preview intent dataset:")
display(dfs["intent_dataset_raw"].head())

print("Preview program studi:")
display(dfs["program_studi_s1_raw"].head())

print("Preview normalisasi bahasa:")
display(dfs["normalisasi_bahasa_raw"].head())

Preview intent dataset:


,utterance_id,utterance,intent_label,bahasa,register,jenjang_user,kelas,minat_keywords,mapel_favorit,hobi_keywords,gaya_belajar,tujuan_karier_keywords,target_program_id,source_type,confidence_label,catatan
0,UTT001,Saya suka matematika dan ingin bekerja di bida...,rekomendasi_prodi,id_formal,formal,SMA,12,matematika|data|analisis,Matematika|Informatika,membuat grafik|menganalisis data,logis|analitis,data analyst,PRG003,seed_manual,0.95,Contoh formal
1,UTT002,"Aku seneng ngoding sama statistik, cocoknya ku...",rekomendasi_prodi,id_nonformal,nonformal,SMK,12,coding|statistik,Matematika|Produktif RPL,ngoding|eksperimen aplikasi,praktik|logis,programmer|data scientist,PRG001,seed_manual,0.90,Contoh non-formal
2,UTT003,"Aku pengin dadi analis data, jurusan sing pas ...",rekomendasi_prodi,jawa_umum,nonformal,SMA,12,data|analisis,Matematika,olah data,analitis,data analyst,PRG003,seed_manual,0.85,Contoh Jawa umum
3,UTT004,"Abdi resep komputer jeung angka, prodi naon nu...",rekomendasi_prodi,sunda_umum,nonformal,SMA,12,komputer|angka,Matematika|Informatika,komputer,logis,software engineer|data analyst,PRG001,seed_manual,0.85,Contoh Sunda umum
4,UTT005,"Kalau ingin masuk Teknik Informatika, saya har...",roadmap_belajar,id_formal,formal,SMA,12,informatika|pemrograman,Informatika,ngoding,praktik,software engineer,PRG001,seed_manual,0.95,Pertanyaan roadmap


Preview program studi:


,program_id,nama_program_studi,jenjang,rumpun_ilmu,bidang_keilmuan,deskripsi_singkat,minat_cocok,mapel_relevan,hobi_relevan,gaya_belajar_cocok,karakteristik_cocok,tingkat_matematika,tingkat_coding,tingkat_komunikasi,tingkat_desain,prospek_karier_id_list,skill_id_list,keyword_rekomendasi,catatan
0,PRG001,Teknik Informatika,S1,Komputer,Software Engineering dan Komputasi,"Mempelajari pemrograman, algoritma, basis data...",coding|logika|komputer|aplikasi,Matematika|Informatika,ngoding|membuat aplikasi|problem solving,praktik|logis|eksperimen,teliti|suka memecahkan masalah,tinggi,tinggi,sedang,rendah,KAR003,SKL001|SKL002|SKL004,informatika|pemrograman|software|aplikasi|algo...,Seed data
1,PRG002,Sistem Informasi,S1,Komputer,Sistem Bisnis dan Teknologi Informasi,"Mempelajari integrasi teknologi, proses bisnis...",bisnis|teknologi|analisis|manajemen,Ekonomi|Informatika|Matematika,organisasi|membuat diagram|analisis proses,komunikatif|analitis|praktik,mampu menjembatani bisnis dan IT,sedang,sedang,tinggi,rendah,KAR005|KAR001,SKL002|SKL003|SKL005,sistem informasi|bisnis|analisis sistem|erp|bu...,Seed data
2,PRG003,Sains Data,S1,Komputer,Data Science dan Analitik,"Mempelajari statistika, pemrograman, machine l...",data|statistik|machine learning|analisis,Matematika|Statistika|Informatika,membuat grafik|menganalisis angka|eksperimen m...,analitis|terstruktur|eksperimen,teliti|suka pola dan data,tinggi,tinggi,sedang,rendah,KAR001|KAR002,SKL001|SKL002|SKL003|SKL006,sains data|data science|statistik|machine lear...,Seed data
3,PRG004,Statistika,S1,Matematika dan Sains,Statistika Terapan,"Mempelajari pengumpulan data, probabilitas, pe...",angka|riset|survei|analisis,Matematika,olah data|membuat survei|riset,analitis|terstruktur,teliti dan kuat secara kuantitatif,tinggi,sedang,sedang,rendah,KAR001|KAR002,SKL003|SKL006,statistika|probabilitas|survei|analisis kuanti...,Seed data
4,PRG005,Desain Komunikasi Visual,S1,Seni dan Desain,Desain Visual dan Komunikasi Digital,"Mempelajari desain grafis, branding, ilustrasi...",desain|visual|kreatif|media,Seni Budaya|TIK,menggambar|editing|membuat konten,visual|kreatif|praktik,kreatif dan peka estetika,rendah,rendah,tinggi,tinggi,KAR004,SKL007,dkv|desain|visual|ui ux|branding|konten,Seed data


Preview normalisasi bahasa:


,norm_id,kata_input,kata_baku,bahasa_sumber,kategori,contoh_kalimat,prioritas,catatan
0,NRM001,gw,saya,id_nonformal,pronomina,gw suka matematika,1,Bahasa gaul umum
1,NRM002,gue,saya,id_nonformal,pronomina,gue bingung pilih jurusan,1,Bahasa gaul umum
2,NRM003,aku,saya,id_nonformal,pronomina,aku mau kuliah apa ya,2,Dipakai untuk variasi non-formal
3,NRM004,pgn,ingin,id_nonformal,singkatan,pgn jadi programmer,1,Singkatan
4,NRM005,pingin,ingin,id_nonformal,variasi,pingin kerja di data,1,Variasi kata


In [20]:
for name, df in dfs.items():
    output_path = DATA_RAW_DIR / f"{name}.csv"
    df.to_csv(output_path, index=False, encoding="utf-8-sig")
    print(f"Berhasil export: {output_path}")

Berhasil export: D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA-MINING\TUGAS\UAS\edupath-career-assitant\data\raw\intent_dataset_raw.csv
Berhasil export: D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA-MINING\TUGAS\UAS\edupath-career-assitant\data\raw\program_studi_s1_raw.csv
Berhasil export: D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA-MINING\TUGAS\UAS\edupath-career-assitant\data\raw\karier_raw.csv
Berhasil export: D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA-MINING\TUGAS\UAS\edupath-career-assitant\data\raw\skill_awal_raw.csv
Berhasil export: D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA-MINING\TUGAS\UAS\edupath-career-assitant\data\raw\roadmap_belajar_raw.csv
Berhasil export: D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA-MINING\TUGAS\UAS\edupath-career-assitant\data\raw\normalisasi_bahasa_raw.csv


In [21]:
loaded_dfs = {}

for file_path in sorted(DATA_RAW_DIR.glob("*_raw.csv")):
    dataset_name = file_path.stem
    loaded_dfs[dataset_name] = pd.read_csv(file_path)
    print(f"{dataset_name}: {loaded_dfs[dataset_name].shape}")

intent_dataset_raw: (12, 16)
karier_raw: (5, 12)
normalisasi_bahasa_raw: (18, 8)
program_studi_s1_raw: (5, 19)
roadmap_belajar_raw: (6, 12)
skill_awal_raw: (7, 12)


## Validasi Dataset

Validasi dilakukan untuk memastikan:
1. Semua kolom wajib tersedia.
2. ID utama tidak kosong.
3. ID utama tidak duplikat.
4. Utterance tidak duplikat.
5. Label intent memiliki distribusi awal yang bisa dipantau.
6. Relasi antar dataset valid.

In [22]:
def split_pipe_values(value):
    """
    Memecah nilai multi-value berbasis delimiter pipe.
    Contoh:
    'KAR001|KAR002' -> ['KAR001', 'KAR002']
    """
    if pd.isna(value) or str(value).strip() == "":
        return []
    return [item.strip() for item in str(value).split("|") if item.strip()]


def add_result(results, dataset, check_name, status, detail):
    """
    Menambahkan hasil validasi ke list.
    """
    results.append({
        "dataset": dataset,
        "check_name": check_name,
        "status": status,
        "detail": detail
    })

In [23]:
validation_results = []

required_columns_map = {
    "intent_dataset_raw": intent_columns,
    "program_studi_s1_raw": program_columns,
    "karier_raw": karier_columns,
    "skill_awal_raw": skill_columns,
    "roadmap_belajar_raw": roadmap_columns,
    "normalisasi_bahasa_raw": normalisasi_columns,
}

id_columns_map = {
    "intent_dataset_raw": "utterance_id",
    "program_studi_s1_raw": "program_id",
    "karier_raw": "karier_id",
    "skill_awal_raw": "skill_id",
    "roadmap_belajar_raw": "roadmap_id",
    "normalisasi_bahasa_raw": "norm_id",
}

# 1. Validasi kolom wajib, ID kosong, dan ID duplikat
for dataset_name, df in loaded_dfs.items():
    required_cols = required_columns_map[dataset_name]
    id_col = id_columns_map[dataset_name]

    missing_cols = [col for col in required_cols if col not in df.columns]
    add_result(
        validation_results,
        dataset_name,
        "required_columns",
        "PASS" if not missing_cols else "FAIL",
        "Semua kolom wajib tersedia." if not missing_cols else f"Kolom belum ada: {missing_cols}"
    )

    empty_id_count = df[id_col].isna().sum() + (df[id_col].astype(str).str.strip() == "").sum()
    add_result(
        validation_results,
        dataset_name,
        "empty_id",
        "PASS" if empty_id_count == 0 else "FAIL",
        f"Jumlah ID kosong: {empty_id_count}"
    )

    duplicate_id_count = df[id_col].duplicated().sum()
    add_result(
        validation_results,
        dataset_name,
        "duplicate_id",
        "PASS" if duplicate_id_count == 0 else "FAIL",
        f"Jumlah ID duplikat: {duplicate_id_count}"
    )

# 2. Validasi utterance duplikat pada dataset intent
intent_df = loaded_dfs["intent_dataset_raw"]

duplicate_utterance_count = intent_df["utterance"].astype(str).str.lower().duplicated().sum()

add_result(
    validation_results,
    "intent_dataset_raw",
    "duplicate_utterance",
    "PASS" if duplicate_utterance_count == 0 else "WARN",
    f"Jumlah utterance duplikat: {duplicate_utterance_count}"
)

# 3. Validasi distribusi label intent
intent_label_counts = intent_df["intent_label"].value_counts()
min_rows_per_label = intent_label_counts.min()

add_result(
    validation_results,
    "intent_dataset_raw",
    "intent_label_distribution",
    "PASS" if min_rows_per_label >= 3 else "WARN",
    f"Minimum data per intent_label: {min_rows_per_label}. Untuk modeling final disarankan >= 100 per label."
)

# 4. Validasi distribusi bahasa
language_counts = intent_df["bahasa"].value_counts()

add_result(
    validation_results,
    "intent_dataset_raw",
    "language_distribution",
    "PASS",
    "Distribusi bahasa: " + "; ".join([f"{idx}={val}" for idx, val in language_counts.items()])
)

# 5. Foreign key: target_program_id di intent harus tersedia pada program_id
program_df = loaded_dfs["program_studi_s1_raw"]
program_ids = set(program_df["program_id"])

intent_target_ids = set(
    item for item in intent_df["target_program_id"].astype(str).str.strip()
    if item != "" and item.lower() != "nan"
)

missing_program_from_intent = sorted(intent_target_ids - program_ids)

add_result(
    validation_results,
    "intent_dataset_raw",
    "fk_target_program_id",
    "PASS" if not missing_program_from_intent else "FAIL",
    "Semua target_program_id tersedia di program_studi_s1_raw."
    if not missing_program_from_intent else f"Program ID belum tersedia: {missing_program_from_intent}"
)

# 6. Foreign key: roadmap.program_id harus tersedia di program_studi
roadmap_df = loaded_dfs["roadmap_belajar_raw"]

roadmap_program_ids = set(roadmap_df["program_id"].astype(str).str.strip())
missing_program_from_roadmap = sorted(roadmap_program_ids - program_ids)

add_result(
    validation_results,
    "roadmap_belajar_raw",
    "fk_program_id",
    "PASS" if not missing_program_from_roadmap else "FAIL",
    "Semua roadmap program_id tersedia di program_studi_s1_raw."
    if not missing_program_from_roadmap else f"Program ID belum tersedia: {missing_program_from_roadmap}"
)

# 7. Foreign key multi-value untuk karier dan skill
karier_df = loaded_dfs["karier_raw"]
skill_df = loaded_dfs["skill_awal_raw"]

career_ids = set(karier_df["karier_id"])
skill_ids = set(skill_df["skill_id"])

program_career_refs = set()
for value in program_df["prospek_karier_id_list"]:
    program_career_refs.update(split_pipe_values(value))

missing_career_refs = sorted(program_career_refs - career_ids)

add_result(
    validation_results,
    "program_studi_s1_raw",
    "fk_prospek_karier_id_list",
    "PASS" if not missing_career_refs else "FAIL",
    "Semua prospek_karier_id_list tersedia di karier_raw."
    if not missing_career_refs else f"Karier ID belum tersedia: {missing_career_refs}"
)

program_skill_refs = set()
for value in program_df["skill_id_list"]:
    program_skill_refs.update(split_pipe_values(value))

missing_skill_refs = sorted(program_skill_refs - skill_ids)

add_result(
    validation_results,
    "program_studi_s1_raw",
    "fk_skill_id_list",
    "PASS" if not missing_skill_refs else "FAIL",
    "Semua skill_id_list tersedia di skill_awal_raw."
    if not missing_skill_refs else f"Skill ID belum tersedia: {missing_skill_refs}"
)

validation_df = pd.DataFrame(validation_results)
display(validation_df)

,dataset,check_name,status,detail
0,intent_dataset_raw,required_columns,PASS,Semua kolom wajib tersedia.
1,intent_dataset_raw,empty_id,PASS,Jumlah ID kosong: 0
2,intent_dataset_raw,duplicate_id,PASS,Jumlah ID duplikat: 0
3,karier_raw,required_columns,PASS,Semua kolom wajib tersedia.
4,karier_raw,empty_id,PASS,Jumlah ID kosong: 0
5,karier_raw,duplicate_id,PASS,Jumlah ID duplikat: 0
6,normalisasi_bahasa_raw,required_columns,PASS,Semua kolom wajib tersedia.
7,normalisasi_bahasa_raw,empty_id,PASS,Jumlah ID kosong: 0
8,normalisasi_bahasa_raw,duplicate_id,PASS,Jumlah ID duplikat: 0
9,program_studi_s1_raw,required_columns,PASS,Semua kolom wajib tersedia.


In [24]:
dataset_overview_path = REPORT_DIR / "dataset_overview.csv"
validation_summary_path = REPORT_DIR / "validation_summary.csv"

dataset_overview_df.to_csv(dataset_overview_path, index=False, encoding="utf-8-sig")
validation_df.to_csv(validation_summary_path, index=False, encoding="utf-8-sig")

print("Laporan dataset berhasil disimpan:")
print(dataset_overview_path)
print(validation_summary_path)

Laporan dataset berhasil disimpan:
D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA-MINING\TUGAS\UAS\edupath-career-assitant\reports\data_quality\dataset_overview.csv
D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA-MINING\TUGAS\UAS\edupath-career-assitant\reports\data_quality\validation_summary.csv


## Preview Normalisasi Bahasa

Bagian ini hanya menunjukkan simulasi sederhana normalisasi teks. Tujuannya adalah memastikan kamus normalisasi bisa digunakan untuk mengubah kata non-formal, Jawa umum, dan Sunda umum ke bentuk yang lebih standar.

In [25]:
normalisasi_df = loaded_dfs["normalisasi_bahasa_raw"]

normalization_map = dict(zip(
    normalisasi_df["kata_input"].astype(str).str.lower(),
    normalisasi_df["kata_baku"].astype(str).str.lower()
))

def normalize_simple_text(text, normalization_map):
    """
    Fungsi normalisasi sederhana berbasis kamus.
    Catatan:
    Ini hanya preview. Preprocessing lengkap dilakukan di Tahap 03.
    """
    text = str(text).lower()

    # Urutkan berdasarkan panjang kata agar frasa lebih panjang diproses lebih dulu
    for kata_input, kata_baku in sorted(normalization_map.items(), key=lambda x: len(x[0]), reverse=True):
        pattern = r"\b" + re.escape(kata_input) + r"\b"
        text = re.sub(pattern, kata_baku, text)

    text = re.sub(r"\s+", " ", text).strip()
    return text

preview_norm = intent_df[["utterance_id", "utterance", "bahasa"]].copy()
preview_norm["utterance_normalized_preview"] = preview_norm["utterance"].apply(
    lambda x: normalize_simple_text(x, normalization_map)
)

display(preview_norm.head(10))

,utterance_id,utterance,bahasa,utterance_normalized_preview
0,UTT001,Saya suka matematika dan ingin bekerja di bida...,id_formal,saya suka matematika dan ingin bekerja di bida...
1,UTT002,"Aku seneng ngoding sama statistik, cocoknya ku...",id_nonformal,"saya suka pemrograman sama statistik, cocoknya..."
2,UTT003,"Aku pengin dadi analis data, jurusan sing pas ...",jawa_umum,"saya ingin menjadi analis data, program studi ..."
3,UTT004,"Abdi resep komputer jeung angka, prodi naon nu...",sunda_umum,"abdi suka komputer jeung angka, prodi apa nu c..."
4,UTT005,"Kalau ingin masuk Teknik Informatika, saya har...",id_formal,"kalau ingin masuk teknik informatika, saya har..."
5,UTT006,Lulusan Sistem Informasi biasanya bisa bekerja...,id_formal,lulusan sistem informasi biasanya bisa bekerja...
6,UTT007,Skill awal untuk jadi data analyst itu apa saja?,id_nonformal,skill awal untuk jadi data analyst itu apa saja?
7,UTT008,Apa perbedaan Teknik Informatika dan Sistem In...,id_formal,apa perbedaan teknik informatika dan sistem in...
8,UTT009,"Halo, saya mau konsultasi jurusan kuliah.",id_formal,"halo, saya mau konsultasi program studi kuliah."
9,UTT010,Saya bingung banget dan belum tahu minat saya ...,id_nonformal,saya bingung banget dan belum tahu minat saya ...


## Export Dataset Interim

Dataset interim adalah versi awal yang sudah distandarkan dari sisi:
1. Spasi berlebih.
2. Nilai teks kosong.
3. Format string dasar.

Dataset interim ini belum melakukan preprocessing NLP penuh seperti case folding final, tokenization, stopword removal, stemming, atau TF-IDF.

In [26]:
def standardize_text_columns(df):
    """
    Membersihkan spasi berlebih pada seluruh kolom bertipe teks.
    """
    df_clean = df.copy()

    for col in df_clean.columns:
        if pd.api.types.is_object_dtype(df_clean[col]) or pd.api.types.is_string_dtype(df_clean[col]):
            df_clean[col] = (
                df_clean[col]
                .fillna("")
                .astype(str)
                .str.replace(r"\s+", " ", regex=True)
                .str.strip()
            )

    return df_clean


interim_dfs = {}

for name, df in loaded_dfs.items():
    interim_name = name.replace("_raw", "_interim")
    interim_dfs[interim_name] = standardize_text_columns(df)

    output_path = DATA_INTERIM_DIR / f"{interim_name}.csv"
    interim_dfs[interim_name].to_csv(output_path, index=False, encoding="utf-8-sig")

    print(f"Berhasil export interim: {output_path}")

Berhasil export interim: D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA-MINING\TUGAS\UAS\edupath-career-assitant\data\interim\intent_dataset_interim.csv
Berhasil export interim: D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA-MINING\TUGAS\UAS\edupath-career-assitant\data\interim\karier_interim.csv
Berhasil export interim: D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA-MINING\TUGAS\UAS\edupath-career-assitant\data\interim\normalisasi_bahasa_interim.csv
Berhasil export interim: D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA-MINING\TUGAS\UAS\edupath-career-assitant\data\interim\program_studi_s1_interim.csv
Berhasil export interim: D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA-MINING\TUGAS\UAS\edupath-career-assitant\data\interim\roadmap_belajar_interim.csv
Berhasil export interim: D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA-MINING\TUGAS\UAS\edupath-career-assitant\data\interim\skill_awal_interim.csv


In [28]:
print("File di data/raw:")
for file_path in sorted(DATA_RAW_DIR.glob("*.csv")):
    print("-", file_path.name)

print("\nFile di data/interim:")
for file_path in sorted(DATA_INTERIM_DIR.glob("*.csv")):
    print("-", file_path.name)

print("\nFile di reports/data_quality:")
for file_path in sorted(REPORT_DIR.glob("*.csv")):
    print("-", file_path.name)

File di data/raw:
- intent_dataset_raw.csv
- karier_raw.csv
- normalisasi_bahasa_raw.csv
- program_studi_s1_raw.csv
- roadmap_belajar_raw.csv
- skill_awal_raw.csv

File di data/interim:
- intent_dataset_interim.csv
- karier_interim.csv
- normalisasi_bahasa_interim.csv
- program_studi_s1_interim.csv
- roadmap_belajar_interim.csv
- skill_awal_interim.csv

File di reports/data_quality:
- dataset_overview.csv
- validation_summary.csv
